In [1]:
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import spacy

logs_compiled = pd.read_csv("Data/logs_compiled.csv")
clicks = pd.read_csv("Data/clicked_results_df.csv")

In [2]:
clicks.head()

,uid,topic_number,qid,query,serp_id,url,rank,relevance_score,timestamp
0,1,g3t3,q1,Quali sono i cinque animali tipici dell india,serp1,https://wwftravel.it/blog/viaggi/india-natura-...,1,1,2026-05-19 14:51:43.444
1,1,g3t3,q1,Quali sono i cinque animali tipici dell india,serp1,https://wwftravel.it/blog/viaggi/india-natura-...,1,1,2026-05-19 14:53:19.408
2,1,g3t2,q2,Quale è il nome del simbolo al centro della ba...,serp2,https://it.gov-civ-guarda.pt/flag-india,1,1,2026-05-19 15:02:48.776
3,1,g3t1,q3,Fiume sacro degli indiani,serp3,https://www.consciousjourneys.com/it/approfond...,2,1,2026-05-19 15:08:58.405
4,2,g3t3,q5,Quali sono i 5 animali tipici dall'India,serp5,https://wwftravel.it/blog/viaggi/india-natura-...,1,1,2026-05-19 14:52:53.759


In [3]:
logs_compiled.head()

,uid,topic_number,qid,query,serp_id,url,title,snippet,rank,relevance_score,click_count,timestamp
0,1,g3t3,q1,Quali sono i cinque animali tipici dell india,serp1,https://wwftravel.it/blog/viaggi/india-natura-...,"India naturalistica, 5 animali da non perdere ...","\n Oct 31, 2023 ... La fauna selvatic...",1.0,1,2,2026-05-19 14:51:27.687
1,1,g3t3,q1,Quali sono i cinque animali tipici dell india,serp1,https://www.terreincognitemagazine.it/natura-s...,"Natura selvaggia in India, tra tigri e leoni -...","\n Feb 17, 2017 ... O dei Giganti, vi...",2.0,1,0,2026-05-19 14:51:27.688
2,1,g3t3,q1,Quali sono i cinque animali tipici dell india,serp1,https://www.storicang.it/a/civilta-dellindo-i-...,"La civiltà dell'Indo, i misteri di un popolo d...","\n Jun 5, 2023 ... Sono note cinque m...",3.0,1,0,2026-05-19 14:51:27.689
3,1,g3t3,q1,Quali sono i cinque animali tipici dell india,serp1,https://www.focusjunior.it/scuola/storia/chi-s...,"Chi sono gli Indiani d'America, i nativi ameri...","\n Jun 6, 2024 ... Chi sono, da dove ...",4.0,0,0,2026-05-19 14:51:27.690
4,1,g3t3,q1,Quali sono i cinque animali tipici dell india,serp1,https://www.focusjunior.it/comportamento/feste...,Festa della mamma: come si festeggia nel resto...,\n Il dolce tipico per celebrare ques...,5.0,0,0,2026-05-19 14:51:27.691


In [4]:
nav_eff_details = []

for serp_id in clicks["serp_id"].unique():
    serp_clicks = clicks.loc[clicks["serp_id"]==serp_id]
    uid = serp_clicks["uid"].values[0]
    topic_number = serp_clicks["topic_number"].values[0]
    qid = serp_clicks["qid"].values[0]
    query = serp_clicks["query"].values[0]
    total_clicks = len(serp_clicks)
    repetitions = total_clicks - len(serp_clicks["url"].unique())
    if len(serp_clicks.loc[serp_clicks["relevance_score"]==1]) == 0:
        last_relevant_timestamp =  None
        tail = len(serp_clicks)
    else:
        last_relevant_timestamp = max(serp_clicks.loc[serp_clicks["relevance_score"]==1]["timestamp"]) 
        tail = len(serp_clicks.loc[serp_clicks["timestamp"]>last_relevant_timestamp])
    
    nav_eff = (total_clicks - repetitions - tail) / total_clicks

    nav_eff_details.append([uid, topic_number, qid, query, serp_id, total_clicks, repetitions, tail, last_relevant_timestamp, nav_eff])

nav_eff_df = pd.DataFrame(nav_eff_details, columns=["uid", "topic_number", "qid", "query", "serp_id", "total_clicks", "repetitions", "tail", "last_relevant_timestamp", "nav_eff"])
nav_eff_df.to_csv("Data/navigation_quality.csv", index=False, encoding="utf8")

In [5]:
nav_eff_df.head()

,uid,topic_number,qid,query,serp_id,total_clicks,repetitions,tail,last_relevant_timestamp,nav_eff
0,1,g3t3,q1,Quali sono i cinque animali tipici dell india,serp1,2,1,0,2026-05-19 14:53:19.408,0.5
1,1,g3t2,q2,Quale è il nome del simbolo al centro della ba...,serp2,1,0,0,2026-05-19 15:02:48.776,1.0
2,1,g3t1,q3,Fiume sacro degli indiani,serp3,1,0,0,2026-05-19 15:08:58.405,1.0
3,2,g3t3,q5,Quali sono i 5 animali tipici dall'India,serp5,1,0,0,2026-05-19 14:52:53.759,1.0
4,2,g3t2,q7,Qual è il nome del simbolo al centro della b...,serp7,1,0,0,2026-05-19 15:09:25.575,1.0


In [6]:
len(nav_eff_df.loc[nav_eff_df["nav_eff"]< 0.8]), len(nav_eff_df.loc[nav_eff_df["nav_eff"]>= 0.8])

(26, 33)

In [7]:
df = pd.DataFrame(logs_compiled.groupby(by=["uid", "serp_id"])["click_count"].sum()).reset_index()
df.loc[df["click_count"]>5]

,uid,serp_id,click_count
56,9,serp57,9
58,9,serp59,9
73,12,serp74,11
78,13,serp79,6
95,15,serp95,10


In [8]:
nav_eff_df.loc[nav_eff_df["serp_id"].isin(df.loc[df["click_count"]>5]["serp_id"].tolist())]

,uid,topic_number,qid,query,serp_id,total_clicks,repetitions,tail,last_relevant_timestamp,nav_eff
24,9,g4t3,q57,Quali sono i nomi di altre 3 galassie oltre la...,serp57,9,5,4,2026-05-19 11:05:36.905,0.000000
25,9,g4t3,q59,3 galassie oltre la nostra,serp59,9,2,2,2026-05-19 11:15:39.675,0.555556
36,12,g4t3,q74,Quali sono i nomi di altre 3 galassie oltre la...,serp74,11,5,1,2026-05-19 11:07:21.821,0.454545
39,13,g4t1,q79,come si chiamano i satelliti di giove,serp79,6,4,0,2026-05-19 10:55:20.453,0.333333
45,15,g4t3,q95,Quali sono i nomi di altre 3 galassie oltre la...,serp95,10,2,3,2026-05-19 11:01:43.835,0.500000


In [9]:
serp_id = "serp95"
print(len(clicks.loc[clicks["serp_id"]==serp_id]))
clicks.loc[clicks["serp_id"]==serp_id]

10


,uid,topic_number,qid,query,serp_id,url,rank,relevance_score,timestamp
128,15,g4t3,q95,Quali sono i nomi di altre 3 galassie oltre la...,serp95,https://www.focusjunior.it/scienza/spazio/stel...,1,0,2026-05-19 10:58:07.868
129,15,g4t3,q95,Quali sono i nomi di altre 3 galassie oltre la...,serp95,https://www.focusjunior.it/scienza/spazio/stel...,1,0,2026-05-19 10:59:20.077
130,15,g4t3,q95,Quali sono i nomi di altre 3 galassie oltre la...,serp95,https://www.castfvg.it/galassie/gruppo_locale/...,3,1,2026-05-19 10:59:49.752
131,15,g4t3,q95,Quali sono i nomi di altre 3 galassie oltre la...,serp95,https://123stella.astronomiamo.it/Fase.aspx?Ar...,2,0,2026-05-19 11:00:13.495
132,15,g4t3,q95,Quali sono i nomi di altre 3 galassie oltre la...,serp95,https://www.focusjunior.it/content/uploads/sit...,4,0,2026-05-19 11:01:17.454
133,15,g4t3,q95,Quali sono i nomi di altre 3 galassie oltre la...,serp95,https://www.focusjunior.it/content/uploads/sit...,4,0,2026-05-19 11:01:24.384
134,15,g4t3,q95,Quali sono i nomi di altre 3 galassie oltre la...,serp95,https://www.focusjunior.it/scienza/spazio/prot...,6,1,2026-05-19 11:01:43.835
135,15,g4t3,q95,Quali sono i nomi di altre 3 galassie oltre la...,serp95,https://www.focusjunior.it/scienza/spazio/mari...,5,0,2026-05-19 11:02:39.838
136,15,g4t3,q95,Quali sono i nomi di altre 3 galassie oltre la...,serp95,https://www.focusjunior.it/content/uploads/201...,7,0,2026-05-19 11:02:47.905
137,15,g4t3,q95,Quali sono i nomi di altre 3 galassie oltre la...,serp95,https://www.focusjunior.it/news/news-e-curiosi...,11,0,2026-05-19 11:03:42.782
